# 02 · Data quality and cleaning
Profile the raw staging tables, choose a rule for each problem, apply it in `src/data_cleaning.py`, and prove the result.

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import get_engine, query, PROJECT_ROOT
from src import analysis as an
an.set_style()
pd.set_option("display.float_format", "{:,.2f}".format); pd.set_option("display.max_columns", 30)
engine = get_engine()
o = an.sql_output

In [2]:
o('dq_a01_row_counts').assign(extra=lambda d: d.raw_rows - d.distinct_keys)

,table_name,raw_rows,distinct_keys,extra
0,campaigns,35523,35523,0
1,complaints,10559,10559,0
2,customers,28576,28496,80
3,subscriptions,31539,31539,0
4,transactions,1775945,1770634,5311
5,usage_monthly,561549,560429,1120


In [3]:
o('dq_a02_label_variants').query('field != "age_group" or n < 400')

,field,value,n
5,age_group,25 - 34,87
6,age_group,18 - 24,75
7,age_group,45 - 54,52
8,age_group,35 - 44,50
9,customer_type,Prepaid,24948
10,customer_type,Postpaid,3377
11,customer_type,PREPAID,76
12,customer_type,prepaid,74
13,customer_type,Prepaid,67
14,customer_type,Postpaid,15


Labels such as `data bundle`, `Data bndl` and `DATA BUNDLE` would split *Data Bundle* revenue into several bars. Each is mapped to a canonical list, and the build **stops** if a label can't be mapped, rather than silently creating a new category.

In [4]:
o('dq_a04_amount_profile')

,band,rows,min_kes,max_kes
0,a) negative (reversal),3550,"-6,689.00",-5.00
1,"b) 1 - 1,000",1707325,5.00,"1,000.00"
2,"c) 1,001 - 15,000",65041,"1,001.00","13,000.00"
3,"d) 15,001 - 25,000",2,"20,000.00","22,000.00"
4,"e) above 25,000",27,"31,000.00","827,000.00"


Legitimate purchases top out around KES 12,300 (large postpaid bills). Everything above 25,000 follows a ×1000 keying pattern. But a single ceiling is not enough: the 15,001–25,000 band holds two rows, and they are **mobile money fees**.

In [5]:
o('dq_a04b_amount_by_service')

,service_type,rows,p99_9_kes,max_kes,rows_above_1000
0,Data Bundle,459162,"3,476.75","792,000.00",7518
1,Mobile Money Fees,547588,95.00,"94,000.00",13
2,Postpaid Bill,77424,"7,696.04","12,290.00",50698
3,Voice Bundle,283653,"3,024.35","827,000.00",2278


Mobile money fees follow a tariff and never exceed KES 95, so a KES 13,000 or 22,000 fee is a KES 13–22 fee keyed ×1000. The rule is therefore **per service**: fees above KES 1,000 and anything above KES 25,000 are **quarantined**. My first version used only the 25,000 ceiling and let three of these through; checking each service's range caught them. Negative amounts are **reversals**: they go to `telco.reversals` instead of being netted silently against revenue.

In [6]:
pd.concat([o('dq_a05_usage_anomalies'), o('dq_a06_date_logic').set_index('check_name').T.reset_index(drop=True)], axis=1)

,negative_data_rows,meter_sentinel_rows,max_plausible_data_mb,acquisition date after data window,complaint resolved before it was raised,transactions for unknown customers
0,25,20,"90,242.30",12,30,46


| Problem | Rule | Why |
|---|---|---|
| Acquisition year typed as 2062 | replace with the first subscription start date | billing activation is a trusted second source, so this is a certain fix, not a guess |
| Negative or 999,999,999 data readings | set `data_mb` to NULL, keep the row | voice and SMS in the same row are valid; the month still counts as active |
| Resolution date before complaint date | set resolution date to NULL, keep status | resolution time is unknowable, but the complaint itself is real |
| Missing campaign response | `Unknown` | so the share without a response stays visible |
| Missing county (3%) | NULL → reported as *Unknown* | imputing a county would be a guess |

In [7]:
o('dq_b03_exception_log')

,source_table,issue,action_taken,records
0,campaigns,missing response,set to 'Unknown',710
1,complaints,category label case,standardised,120
2,complaints,status label case,standardised,90
3,complaints,resolution date before complaint date,resolution_date set to NULL; status kept,30
4,customers,missing county,left NULL (reported as Unknown),854
5,customers,age group label with spaces,standardised,264
6,customers,customer type label variant,standardised,250
7,customers,exact duplicate row,removed duplicate copy,80
8,customers,acquisition date outside data range (year typo),replaced with first subscription start,12
9,transactions,service type label variant,mapped to canonical service,32509


## Proof: clean-layer checks (all must be 0)

In [8]:
chk = o('dq_b01_integrity_summary'); assert chk.failures.sum() == 0, chk; chk

,check_name,failures
0,duplicate customer-month usage rows,0
1,customers with no subscription row,0
2,customers with more than one open subscription,0
3,customers with more than one churn record,0
4,transactions before acquisition,0
5,transactions after line closed,0
6,acquisition date not equal to first subscripti...,0
7,customer-month grid rows with activity after c...,0


In [9]:
o('dq_b02_grid_reconciliation')

,revenue_in_transactions,revenue_in_grid,usage_rows,usage_rows_matched_in_grid
0,"454,997,387.00","454,997,387.00",560429,560429


Revenue in the customer-month grid equals revenue in the transactions table to the shilling, and every usage row lands in exactly one grid cell. Downstream ARPU and churn figures therefore rest on complete data.